# Time diagnostics
Create one PDF for energy and enstrophy and another for their drag and viscous dissipation rates. Select a time or frame interval and optionally apply a centered rolling average.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt

SCRIPT_DIRECTORY = Path.cwd() if (Path.cwd() / 'ns2d_plotting.py').exists() else Path.cwd() / 'scripts'
sys.path.insert(0, str(SCRIPT_DIRECTORY.resolve()))
from ns2d_plotting import (domain_lengths, filter_time_series, read_csv,
    read_parameters, repository_root, rolling_mean, save_figure,
    use_plot_style)

## Configuration

In [ ]:
ROOT = repository_root()
DIAGNOSTICS_FILE = ROOT / 'output/diagnostics.csv'
PARAMETER_FILE = ROOT / 'output/resolved_parameters.txt'
QUANTITIES_FIGURE = ROOT / 'figures/diagnostics_quantities.pdf'
DISSIPATION_FIGURE = ROOT / 'figures/diagnostics_dissipation.pdf'

# Inclusive time interval, e.g. (0.1, 1.0). None keeps every time.
# Either bound may be None, e.g. (0.1, None).
TIME_RANGE = None
# Inclusive frame interval, e.g. (100, 1000). None keeps every frame.
FRAME_RANGE = None
# Number of saved samples in the centered moving average; 1 plots raw data.
ROLLING_WINDOW = 1
# Divide quantities and rates by the physical domain area.
NORMALIZE_BY_AREA = False
USE_TEX = True
FONT_SIZE = 15

In [ ]:
use_plot_style(USE_TEX, FONT_SIZE)
table = filter_time_series(read_csv(DIAGNOSTICS_FILE),
                           TIME_RANGE, FRAME_RANGE)
parameters = read_parameters(PARAMETER_FILE) if PARAMETER_FILE.exists() else {}
lx, ly = domain_lengths(parameters)
normalization = lx * ly if NORMALIZE_BY_AREA else 1.0
time = table['time']

def series(column):
    return rolling_mean(table[column] / normalization, ROLLING_WINDOW)

print(f'Using {table.size} samples, frames '      f'{int(table["frame"][0])}--{int(table["frame"][-1])}')

## Energy and enstrophy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].plot(time, series('energy'), label=r'$E$')
axes[1].plot(time, series('enstrophy'), label=r'$Z$', color='C1')
axes[0].set_ylabel('energy density' if NORMALIZE_BY_AREA else 'energy')
axes[1].set_ylabel('enstrophy density' if NORMALIZE_BY_AREA else 'enstrophy')
for axis in axes:
    axis.set_xlabel(r'$t$')
    axis.grid(True, alpha=0.2)
    axis.legend()
saved = save_figure(fig, QUANTITIES_FIGURE)
print(f'Wrote {saved}')
plt.show()

## Dissipation rates

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for axis, quantity, symbol in zip(axes, ['energy', 'enstrophy'], ['E', 'Z']):
    drag = series(f'{quantity}_dissipation_drag')
    viscosity = series(f'{quantity}_dissipation_viscosity')
    axis.plot(time, drag, label='drag')
    axis.plot(time, viscosity, label='viscosity')
    axis.plot(time, drag + viscosity, label='total',
              color='black', linestyle='--')
    axis.set_xlabel(r'$t$')
    axis.set_ylabel(rf'dissipation rate of ${symbol}$')
    axis.grid(True, alpha=0.2)
    axis.legend()
saved = save_figure(fig, DISSIPATION_FIGURE)
print(f'Wrote {saved}')
plt.show()